In [2]:
import os
import json
import random
from pathlib import Path
from PIL import Image
from unsloth import FastVisionModel
import torch
from unsloth import is_bf16_supported
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Failed editing tqdm to replace Inductor Compilation:


Exception: cannot import name 'AttrsDescriptor' from 'triton.compiler.compiler' (c:\Users\Admin\Desktop\venv\Lib\site-packages\triton\compiler\compiler.py)

In [5]:
# ── Tool migration note ───────────────────────────────────────────────────────
# search_email replaces: check_email_received, verify_email_content,
#   verify_email_sender, get_email_link, verify_no_email
# Email records do NOT require screenshots or DOM — text-only training is valid.
# ─────────────────────────────────────────────────────────────────────────────
MODEL_NAME       = "sanaX3065/Orvion-vl-3b"       # your existing HF model
DATASET_FILE     = r"datasets/executor/aegis_generated_fixed.jsonl"   # browser-specific only
PERFECT_FILE     = None                             # skip — already in the model

OUTPUT_LORA_DIR  = "./orvion_lora"
OUTPUT_MODEL_DIR = "./orvionaegisv1"             # merged weights saved here
LOG_DIR          = "./logs"

# ── Hyperparams ───────────────────────────────────────────────────────────────
MAX_SEQ_LENGTH = 2048    # covers all records safely (max ~811 tokens)
LORA_RANK        = 16      # same rank keeps adapter lightweight
LORA_ALPHA       = 16      # 1:1 ratio for continued fine-tune — less aggressive
LORA_DROPOUT     = 0.05
BATCH_SIZE       = 1       # safest for VL + images on 12GB
GRAD_ACCUM       = 8       # effective batch = 8
EPOCHS           = 2       # 2 is enough — format already learned
LEARNING_RATE    = 5e-5    # lower than fresh training (2e-4) — preserves existing knowledge
WARMUP_STEPS     = 20
WEIGHT_DECAY     = 0.01
LR_SCHEDULER     = "cosine"
SAVE_STEPS       = 100     # checkpoint every 100 steps — safe to resume if crash
LOGGING_STEPS    = 10
SEED             = 42

True

In [ ]:
os.path.exists(r'datasets/executor/generated_screenshots/neg_expand_number_input_wrong_selector_step1_1772382161.png')

True

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# DATASET LOADING
# ─────────────────────────────────────────────────────────────────────────────

def load_dataset(dataset_file: str) -> list:
    records = []
    with open(dataset_file) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))

    random.seed(SEED)
    random.shuffle(records)

    print(f"  Loaded {len(records)} records from {dataset_file}")
    return records


# ─────────────────────────────────────────────────────────────────────────────
# IMAGE LOADING
# ─────────────────────────────────────────────────────────────────────────────

def load_image(image_path: str) -> Image.Image | None:
    path = Path(r"datasets/executor/"+image_path)
    if not path.exists():
        # Try normalising Windows backslashes to forward slashes
        path = Path(image_path.replace("\\", "/"))
    if not path.exists():
        return None
    try:
        img = Image.open(path).convert("RGB")
        # Cap at 1280x800 — keeps VRAM usage stable
        if img.width > 1280 or img.height > 800:
            img.thumbnail((1280, 800), Image.LANCZOS)
        return img
    except Exception:
        return None

# ─────────────────────────────────────────────────────────────────────────────
# RECORD FORMATTING
# ─────────────────────────────────────────────────────────────────────────────

def format_record(record: dict) -> dict | None:
    """
    Convert raw JSONL record → {"messages": [...], "images": [PIL.Image]}

    Two record types are supported:
      ① Visual records   — contain "image" blocks (browser screenshots + DOM)
                           → images list is populated; Unsloth vision collator handles them.
      ② Text-only records — NO image blocks (e.g. email/search_email flows)
                           → images list stays empty; record is still kept.

    Returns None only on a HARD failure (image block present but file missing).
    """
    messages = record.get("messages", [])
    images = []
    formatted_messages = []

    for msg in messages:
        role = msg["role"]
        content = msg["content"]

        if isinstance(content, str):
            formatted_messages.append({"role": role, "content": content})
            continue

        parts = []
        has_image_block = any(b.get("type") == "image" for b in content)

        for block in content:
            if block.get("type") == "text":
                parts.append({"type": "text", "text": block["text"]})

            elif block.get("type") == "image":
                img = load_image(block.get("image", ""))
                if img is None:
                    # Image block declared but file is missing — hard skip
                    return None
                images.append(img)
                parts.append({"type": "image"})

        formatted_messages.append({"role": role, "content": parts})

    # ── Text-only record (email flows, pure-logic steps) ──────────────────
    # No image blocks at all → valid, keep as-is with empty images list.
    # The UnslothVisionDataCollator handles empty image lists gracefully.
    if not images:
        return {"messages": formatted_messages, "images": []}

    return {"messages": formatted_messages, "images": images}



In [ ]:
print("\n" + "="*60)
print("  Orvion-VL Continued Fine-tune")
print(f"  Base:    {MODEL_NAME}")
print(f"  Data:    {DATASET_FILE}")
print(f"  Output:  {OUTPUT_MODEL_DIR}")
print("="*60 + "\n")


  Orvion-VL Continued Fine-tune
  Base:    sanaX3065/Orvion-vl-3b
  Data:    datasets/executor/aegis_generated_fixed.jsonl
  Output:  ./orvion_aegis_v2



In [ ]:
print("Step 1: Loading model from HuggingFace...")


model, tokenizer = FastVisionModel.from_pretrained(
    model_name      = MODEL_NAME,
    max_seq_length  = MAX_SEQ_LENGTH,
    dtype           = None,        # auto — bf16 on 5070 Ti (Blackwell)
    load_in_4bit    = True,        # QLoRA 4-bit
)
print(f"  ✓ Model loaded")

# ── 2. Apply LoRA ──────────────────────────────────────────────
print("Step 2: Applying LoRA adapters...")
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r                          = LORA_RANK,
    lora_alpha                 = LORA_ALPHA,
    lora_dropout               = LORA_DROPOUT,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",   # saves ~30% VRAM
    random_state               = SEED,
)

Step 1: Loading model from HuggingFace...
==((====))==  Unsloth 2026.2.1: Fast Qwen2_5_Vl patching. Transformers: 4.57.6.
   \\   /|    NVIDIA GeForce RTX 5070 Ti Laptop GPU. Num GPUs = 1. Max memory: 11.94 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.12.0.dev20260221+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]c:\Users\prasasnna\Miniconda3\envs\blackwell\lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading checkpoint shards: 100%|██████████| 2/2 [00:12<00:00,  6.01s/it]


  ✓ Model loaded
Step 2: Applying LoRA adapters...


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


In [ ]:
torch.cuda.empty_cache()

In [ ]:
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"  ✓ Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

  ✓ Trainable: 41,084,928 / 2,093,459,456 (1.96%)


In [ ]:
print("
Step 3: Preparing dataset...")

# EMAIL_DATASET_FILE is optional — set to path of email dataset or None to skip
EMAIL_DATASET_FILE = "datasets/executor/aegis_email_dataset.jsonl"  # set None to skip

all_raw = load_dataset(DATASET_FILE)
if EMAIL_DATASET_FILE:
    import os
    if os.path.exists(EMAIL_DATASET_FILE):
        email_raw = load_dataset(EMAIL_DATASET_FILE)
        all_raw.extend(email_raw)
        random.shuffle(all_raw)
        print(f"  + Email dataset merged ({len(email_raw)} records)")
    else:
        print(f"  ⚠ Email dataset not found at {EMAIL_DATASET_FILE} — skipping")

formatted = []
skipped_missing_img = 0
text_only_count = 0

for rec in all_raw:
    result = format_record(rec)
    if result is None:
        skipped_missing_img += 1
    else:
        if not result["images"]:
            text_only_count += 1
        formatted.append(result)

visual_count = len(formatted) - text_only_count
print(f"  ✓ {len(formatted)} records ready")
print(f"    → {visual_count} visual (screenshot + DOM)")
print(f"    → {text_only_count} text-only (email / headless flows)")
print(f"    → {skipped_missing_img} hard-skipped (image block declared but file missing)")

if len(formatted) == 0:
    print("
❌ No valid records found.")

from datasets import Dataset
hf_dataset = Dataset.from_list(formatted)


Step 3: Preparing dataset...
  Loaded 854 records from datasets/executor/aegis_generated_fixed.jsonl
  ✓ 854 records ready  (0 skipped — missing screenshots)


In [ ]:
print("\nStep 4: Setting up trainer...")
Path(LOG_DIR).mkdir(exist_ok=True)
Path(OUTPUT_LORA_DIR).mkdir(exist_ok=True)


Step 4: Setting up trainer...


In [ ]:
from transformers import AutoProcessor
from PIL import Image
import torch

processor = AutoProcessor.from_pretrained("sanaX3065/Orvion-vl-3b")
img = Image.open(r"datasets/executor/generated_screenshots/blocked_expand_checkbox_form_missing_step1_1772382133.png")

# pixel_values shape tells you the real patch count
inputs = processor(images=img, text="test", return_tensors="pt")
pixel_values = inputs["pixel_values"]  # [num_patches, patch_dim]

num_patches = pixel_values.shape[0]  # 5336 from earlier

# Qwen2.5-VL compresses patches via its MLP projector — typically 4:1 compression
# so 5336 patches → ~1334 visual tokens fed into the LM
visual_tokens_after_compression = num_patches // 4
print(f"Raw patches:          {num_patches}")
print(f"Visual tokens (~4:1): {visual_tokens_after_compression}")
print(f"MAX_SEQ_LENGTH:       1024")
print(f"Headroom for text:    {1024 - visual_tokens_after_compression}")

Raw patches:          5336
Visual tokens (~4:1): 1334
MAX_SEQ_LENGTH:       1024
Headroom for text:    -310


In [ ]:
trainer = SFTTrainer(
        model          = model,
        tokenizer      = tokenizer,
        data_collator  = UnslothVisionDataCollator(model, tokenizer),
        train_dataset  = hf_dataset,
        args           = SFTConfig(
            per_device_train_batch_size  = BATCH_SIZE,
            gradient_accumulation_steps  = GRAD_ACCUM,
            warmup_steps                 = WARMUP_STEPS,
            num_train_epochs             = EPOCHS,
            learning_rate                = LEARNING_RATE,
            fp16                         = not is_bf16_supported(),
            bf16                         = is_bf16_supported(),
            logging_steps                = LOGGING_STEPS,
            save_steps                   = SAVE_STEPS,
            save_total_limit             = 2,
            output_dir                   = OUTPUT_LORA_DIR,
            logging_dir                  = LOG_DIR,
            optim                        = "adamw_8bit",
            weight_decay                 = WEIGHT_DECAY,
            lr_scheduler_type            = LR_SCHEDULER,
            seed                         = SEED,
            remove_unused_columns        = False,
            report_to                    = "none",
            dataloader_num_workers       = 0,
            dataset_text_field           = "",
            dataset_kwargs               = {"skip_prepare_dataset": True},
            max_seq_length               = MAX_SEQ_LENGTH,
        ),
    )

Unsloth: Model does not have a default image size - using 512


In [ ]:
import os

# Disable Unsloth’s custom compiler kernels and xFormers entirely
os.environ["UNSLOTH_FORCE_DISABLE_COMPILER"] = "1"
os.environ["XFORMERS_DISABLED"] = "1"

# Optional: ensure PyTorch uses SDPA attention
os.environ["PYTORCH_ENABLE_SDPA"] = "1"
os.environ["PYTORCH_USE_FLASH_ATTENTION"] = "0"
os.environ["DISABLE_TRITON"] = "1"


In [ ]:
# ── 5. Train ───────────────────────────────────────────────────
steps_per_epoch = len(formatted) // (BATCH_SIZE * GRAD_ACCUM)
total_steps     = steps_per_epoch * EPOCHS
print(f"\nStep 5: Training...")
print(f"  Records:         {len(formatted)}")
print(f"  Epochs:          {EPOCHS}")
print(f"  Steps/epoch:     ~{steps_per_epoch}")
print(f"  Total steps:     ~{total_steps}")
print(f"  Effective batch: {BATCH_SIZE * GRAD_ACCUM}")
print(f"  Learning rate:   {LEARNING_RATE} (continued fine-tune)")
print(f"  Est. time:       ~{total_steps * 4 // 60} min on RTX 5070 Ti\n")

stats = trainer.train()


Step 5: Training...
  Records:         854
  Epochs:          2
  Steps/epoch:     ~106
  Total steps:     ~212
  Effective batch: 8
  Learning rate:   5e-05 (continued fine-tune)
  Est. time:       ~14 min on RTX 5070 Ti



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 854 | Num Epochs = 2 | Total steps = 214
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 41,084,928 of 3,795,707,904 (1.08% trained)
c:\Users\prasasnna\Miniconda3\envs\blackwell\lib\site-packages\bitsandbytes\_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
c:\Users\prasasnna\Miniconda3\envs\blackwell\lib\site-packages\bitsandbytes\_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
c:\Users\prasasnna\Miniconda3\envs\blackwell\lib\site-packages\bitsandbytes\_ops.py:239: FutureWarning: _chec

Step,Training Loss
10,2.050800
20,1.763200
30,1.248000
40,0.817800
50,0.576500
60,0.441000
70,0.426800
80,0.357200
90,0.311500
100,0.270700


In [ ]:
# ── 6. Save LoRA ───────────────────────────────────────────────
print("\nStep 6: Saving LoRA adapter...")
model.save_pretrained(OUTPUT_LORA_DIR)
tokenizer.save_pretrained(OUTPUT_LORA_DIR)
print(f"  ✓ LoRA adapter → {OUTPUT_LORA_DIR}")

# ── 7. Merge LoRA into model weights ───────────────────────────
print("\nStep 7: Merging LoRA weights into model (bf16)...")
print(f"  This takes 3-5 minutes and needs ~6GB free VRAM...")
Path(OUTPUT_MODEL_DIR).mkdir(exist_ok=True)

model.save_pretrained_merged(
    OUTPUT_MODEL_DIR,
    tokenizer,
    save_method = "merged_16bit",   # bf16 merged weights — ready to deploy
)
print(f"  ✓ Merged model → {OUTPUT_MODEL_DIR}")

# ── 8. Done ────────────────────────────────────────────────────
runtime_min = stats.metrics.get("train_runtime", 0) / 60
print("\n" + "="*60)
print("  ✅ Done!")
print(f"  Training time:  {runtime_min:.1f} min")
print(f"  Final loss:     {stats.metrics.get('train_loss', 0):.4f}")
print(f"  LoRA adapter:   {OUTPUT_LORA_DIR}")
print(f"  Merged model:   {OUTPUT_MODEL_DIR}")
print("="*60)


Step 6: Saving LoRA adapter...
  ✓ LoRA adapter → ./orvion_lora

Step 7: Merging LoRA weights into model (bf16)...
  This takes 3-5 minutes and needs ~6GB free VRAM...
Found HuggingFace hub cache directory: C:\Users\prasasnna\.cache\huggingface\hub
Checking cache directory for required files...


Unsloth: Copying 2 files from cache to `./orvion_aegis_v2`: 100%|██████████| 2/2 [00:04<00:00,  2.29s/it]


Successfully copied all 2 files from cache to `./orvion_aegis_v2`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [00:14<00:00,  7.17s/it]


Unsloth: Merge process complete. Saved to `d:\orvion\finetuning\orvion_aegis_v2`
  ✓ Merged model → ./orvion_aegis_v2

  ✅ Done!
  Training time:  30.3 min
  Final loss:     0.4915
  LoRA adapter:   ./orvion_lora
  Merged model:   ./orvion_aegis_v2


In [1]:
import torch
import transformers

print(torch.__version__)
print(transformers.__version__)


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\Admin\Desktop\venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\Admin\Desktop\venv\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\Admin\Desktop\venv\Lib\site-packages\ipykernel\kernelapp.py", line 739, in start
    self.io_loop.start()
  File "c:\Users\Admin\Desktop\venv\L

2.2.2+cu121
4.57.0


In [1]:
from transformers import AutoModelForVision2Seq, AutoProcessor
OUTPUT_MODEL_DIR = "./orvionaegisv1"
repo_id = "sanaX3065/Orvion-vl-3b"

model = AutoModelForVision2Seq.from_pretrained("{OUTPUT_MODEL_DIR}")
processor = AutoProcessor.from_pretrained("{OUTPUT_MODEL_DIR}")

model.push_to_hub(repo_id)
processor.push_to_hub(repo_id)

Skipping import of cpp extensions due to incompatible torch version 2.6.0+cu124 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info


ImportError: cannot import name 'AttrsDescriptor' from 'triton.compiler.compiler' (c:\Users\Admin\Desktop\venv\Lib\site-packages\triton\compiler\compiler.py)

In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
from PIL import Image
import torch


model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    OUTPUT_MODEL_DIR,
    torch_dtype=torch.bfloat16,
    device_map="cuda"
)
model.eval()

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:07<00:00,  3.57s/it]


Qwen2_5_VLForConditionalGeneration(
  (model): Qwen2_5_VLModel(
    (visual): Qwen2_5_VisionTransformerPretrainedModel(
      (patch_embed): Qwen2_5_VisionPatchEmbed(
        (proj): Conv3d(3, 1280, kernel_size=(2, 14, 14), stride=(2, 14, 14), bias=False)
      )
      (rotary_pos_emb): Qwen2_5_VisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-31): 32 x Qwen2_5_VLVisionBlock(
          (norm1): Qwen2RMSNorm((1280,), eps=1e-06)
          (norm2): Qwen2RMSNorm((1280,), eps=1e-06)
          (attn): Qwen2_5_VLVisionAttention(
            (qkv): Linear(in_features=1280, out_features=3840, bias=True)
            (proj): Linear(in_features=1280, out_features=1280, bias=True)
          )
          (mlp): Qwen2_5_VLMLP(
            (gate_proj): Linear(in_features=1280, out_features=3420, bias=True)
            (up_proj): Linear(in_features=1280, out_features=3420, bias=True)
            (down_proj): Linear(in_features=3420, out_features=1280, bias=True)
            (act_fn): SiLUAc

In [ ]:
processor = AutoProcessor.from_pretrained("sanaX3065/Orvion-vl-3b")

In [ ]:
def test_inference(screenshot_path, task, dom=None, prev_action=None):
    img = Image.open(screenshot_path).convert("RGB")

    system = (
        "You are Aegis. Current Mode: QUALITY_TESTER\n"
        "Tools: [click, type, clear_and_type, open_url, verify_element_visible, "
        "verify_input_value, verify_text_present, verify_url_contains, "
        "raise_bug_ticket, mark_step_pass, mark_flow_blocked]"
    )

    # Build context block exactly like training data
    dom_str = dom if dom else "[]"
    obs_str = prev_action if prev_action else "None"
    context = f"<CONTEXT_BLOCK>\n[DOM]:\n{dom_str}\n[OBSERVATIONS]:\n{obs_str}\n</CONTEXT_BLOCK>\n\n{task}"

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": [
            {"type": "image", "image": img},
            {"type": "text",  "text": context}
        ]},
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text], images=image_inputs, videos=video_inputs,
        return_tensors="pt", padding=True
    ).to("cuda")

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=256, temperature=0.1, do_sample=True)

    response = processor.batch_decode(
        output[:, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0]
    print(f"\n{'='*60}")
    print(f"[Task]: {task[:80]}")
    print(f"[Response]:\n{response}")
    return response

In [ ]:
# TEST 1 — Basic action: type into field
test_inference(
    r"datasets/executor/generated_screenshots/demoqa_textbox_form_step2_1772371745.png",
    "Enter John Doe in the Full Name field."
)
# Expected: clear_and_type on #userName

# TEST 2 — Verification: check URL after login
test_inference(
    r"datasets/executor/generated_screenshots/saucedemo_login_step7_1772371751.png",
    "Verify that the URL now contains 'inventory', confirming successful login."
)
# Expected: VISUAL_OBSERVATION + verify_url_contains + [GOAL ACHIEVED]

# TEST 3 — Verification: element visible
test_inference(
    r"datasets/executor/generated_screenshots/saucedemo_login_step8_1772371752.png",
    "Verify that the inventory product list is visible after login."
)
# Expected: VISUAL_OBSERVATION + verify_element_visible + [GOAL ACHIEVED]

# TEST 4 — Bug raise + flow blocked
test_inference(
    r"datasets/executor/generated_screenshots/element_not_found_negative_step2_1772371837.png",
    "Verify if element #nonExistentElement is visible on the page."
)
# Expected: VISUAL_OBSERVATION + raise_bug_ticket + [FLOW BLOCKED]

# TEST 5 — Scroll action
test_inference(
    r"datasets/executor/generated_screenshots/scroll_to_submit_button_step2_1772373226.png",
    "Scroll the page to bring the Submit button into the visible viewport."
)
# Expected: scroll_to_element on #submit

# TEST 6 — Dropdown selection
test_inference(
    r"datasets/executor/generated_screenshots/demoqa_select_menu_step3_1772371758.png",
    "Select 'Red' from the Old Style Select Menu dropdown."
)



[Task]: Enter John Doe in the Full Name field.
[Response]:
Thought: I need to clear any existing value in the Full Name field and type "John Doe". The field is visible and interactive.
Action: {"tool": "clear_and_type", "args": {"selector": "#full-name", "text": "John Doe"}}
Final Answer: Typed "John Doe" into Full Name field.


[Task]: Verify that the URL now contains 'inventory', confirming successful login.
[Response]:
Thought: The current URL contains 'inventory', which verifies the expected path after login. This is a success check.
Action: {"tool": "verify_url_contains", "args": {"substring": "inventory"}}
Final Answer: Verified: URL contains "inventory". Step passes.
[GOAL ACHIEVED]


[Task]: Verify that the inventory product list is visible after login.
[Response]:
Thought: I need to verify the presence of the inventory product list. The main heading "Products" is a strong indicator of the page's current state.
Action: verify_text_present("Products")
Final Answer: Step 1 passe

'Thought: I need to verify the Old Style Select Menu field is visible and contains the value \'Red\'. The field is present in the DOM and contains the value \'Red\'.\nAction: {"tool": "verify_element_visible", "args": {"selector": "select#old-style-select-menu"}}\nFinal Answer: Verified: element #old-style-select-menu is visible.\nStep 5: {"tool": "verify_input_value", "args": {"selector": "#old-style-select-menu", "expected": "Red"}}'

In [ ]:
dom_context = """<CONTEXT_BLOCK>
[DOM]:
[{"tag": "input", "selector": "#userName", "type": "text", "value": "", "text": "", "in_viewport": true}]
[OBSERVATIONS]:
None
</CONTEXT_BLOCK>"""
# ── STRESS TEST 1 — scroll_to_element (failed before, used wrong tool) ──────
# Must use scroll_to_element with #submit selector, NOT scroll_to or scroll_down
test_inference(
    r"datasets/executor/generated_screenshots/scroll_to_submit_button_step2_1772373226.png",
    dom_context + "Scroll the page to bring the Submit button into the visible viewport."
)

# ── STRESS TEST 2 — select_option (confused with verify before) ──────────────
# Must use select_option on #oldSelectMenu with value Red, NOT verify anything
test_inference(
    r"datasets/executor/generated_screenshots/demoqa_select_menu_step3_1772371758.png",
    dom_context + "Task - Select 'Red' from the Old Style Select Menu dropdown"
)

# ── STRESS TEST 3 — wrong selector must raise bug, not hallucinate ────────────
# Must call raise_bug_ticket. Must NOT invent a working selector.
test_inference(
    r"datasets/executor/generated_screenshots/neg_selector_wrong_id_step2_1772373166.png",
    dom_context + "Verify element #fullName is present. NOTE: this selector does NOT exist on this page."
)

# ── STRESS TEST 4 — verify_input_value with VISUAL_OBSERVATION ───────────────
# Must start with VISUAL_OBSERVATION, use verify_input_value, end with [GOAL ACHIEVED]
test_inference(
    r"datasets/executor/generated_screenshots/demoqa_textbox_form_step3_1772371745.png",
    dom_context + "Validate that the Full Name field contains John Doe."
)

# ── STRESS TEST 5 — double_click (rare tool, low training count) ─────────────
# Must use double_click on #doubleClickBtn, not regular click
test_inference(
    r"datasets/executor/generated_screenshots/demoqa_buttons_step2_1772371799.png",
    dom_context + "Double-click the Double Click button."
)

# ── STRESS TEST 6 — press_key ────────────────────────────────────────────────
# Must use press_key with key=Enter
test_inference(
    r"datasets/executor/generated_screenshots/herald_key_press_enter_step3_1772380721.png",
    dom_context + "Press the Enter key while the input field is focused."
)

# ── STRESS TEST 7 — mark_step_pass at end of flow ────────────────────────────
# Must use mark_step_pass with a message, not just say step passed in text
test_inference(
    r"datasets/executor/generated_screenshots/demoqa_textbox_form_step10_1772371748.png",
    dom_context + "Mark the Text Box form flow as passed."
)

# ── STRESS TEST 8 — ambiguous task, model must pick right verify tool ─────────
# Deliberately vague — does it pick verify_text_present or verify_element_visible?
test_inference(
    r"datasets/executor/generated_screenshots/demoqa_textbox_form_step9_1772371747.png",
    dom_context + "Check that the submitted output section shows John Doe."
)

# ── STRESS TEST 9 — hallucination trap: plausible but wrong selector ──────────
# The selector looks real but is wrong casing — must raise bug, not hallucinate fix
test_inference(
    r"datasets/executor/generated_screenshots/demoqa_textbox_form_step2_1772371745.png",
    dom_context + "Verify element #FullName is present on this page. The selector may be incorrect."
)

# ── STRESS TEST 10 — completely unseen task phrasing ─────────────────────────
# Same screenshot as test 1 but rephrased — does it generalise or pattern match?
test_inference(
    r"datasets/executor/generated_screenshots/demoqa_textbox_form_step2_1772371745.png",
    dom_context + "The QA spec requires typing 'John Doe' into the name input. Execute this step."
)


[Task]: <CONTEXT_BLOCK>
[DOM]:
[{"tag": "input", "selector": "#userName", "type": "text", "value": "", "text": "", "in_viewport": true}]
[OBSERVATIONS]:
None
</CONTEXT_BLOCK>Scroll the page to bring the Submit button into the visible viewport.
[Response]:
Thought: The Submit button is not visible in the current viewport. I will scroll it into view.
Action: {"tool": "scroll_to", "args": {"direction": "bottom"}}
Final Answer: Scrolled to bring the Submit button into view.
[GOAL ACHIEVED]


[Task]: <CONTEXT_BLOCK>
[DOM]:
[{"tag": "input", "selector": "#userName", "type": "text", "value": "", "text": "", "in_viewport": true}]
[OBSERVATIONS]:
None
</CONTEXT_BLOCK>Task - Select 'Red' from the Old Style Select Menu dropdown
[Response]:
Thought: The field #color is visible and enabled. I will clear any existing value and type 'Red'.
Action: {"tool": "clear_and_type", "args": {"selector": "#color", "text": "Red"}}
Final Answer: Typed 'Red' into field #color. Awaiting updated screenshot and DOM

'Thought: I need to type \'John Doe\' into the input field. The field is visible and ready for input.\nAction: {"tool": "clear_and_type", "args": {"selector": "#userName", "text": "John Doe"}}\nFinal Answer: Typed \'John Doe\' into input field.'

In [ ]:
# ── [GOAL ACHIEVED] TESTS ────────────────────────────────────────────────────

# GA-1: verify_input_value — DOM has the value, must match and close with [GOAL ACHIEVED]
test_inference(
    r"datasets/executor/generated_screenshots/demoqa_textbox_form_step3_1772371745.png",
    "Validate that the Full Name field contains John Doe.",
    dom='[{"tag": "input", "selector": "#userName", "type": "text", "value": "John Doe", "text": "John Doe", "in_viewport": true}]'
)

# GA-2: verify_input_value — different field, email
test_inference(
    r"datasets/executor/generated_screenshots/demoqa_textbox_form_step5_1772371746.png",
    "Validate that the Email field contains john.doe@testmail.com.",
    dom='[{"tag": "input", "selector": "#userEmail", "type": "email", "value": "john.doe@testmail.com", "text": "john.doe@testmail.com", "in_viewport": true}]'
)

# GA-3: verify_text_present — no selector needed, just visual scan
test_inference(
    r"datasets/executor/generated_screenshots/demoqa_textbox_form_step9_1772371747.png",
    "Verify that the submitted output section shows John Doe.",
    dom='[{"tag": "a", "selector": "a.router-link", "text": "Text Box", "type": "", "value": "", "in_viewport": true}]'
)

# GA-4: verify_url_contains — after login, URL check
test_inference(
    r"datasets/executor/generated_screenshots/saucedemo_login_step7_1772371751.png",
    "Verify that the URL now contains 'inventory', confirming successful login.",
    dom='[{"tag": "input", "selector": "[data-test=\"username\"]", "type": "text", "value": "standard_user", "in_viewport": true}]'
)

# GA-5: mark_step_pass — full flow completed
test_inference(
    r"datasets/executor/generated_screenshots/demoqa_textbox_form_step10_1772371748.png",
    "Mark the Text Box form flow as passed.",
    dom='[{"tag": "a", "selector": "a.router-link", "text": "Text Box", "type": "", "value": "", "in_viewport": true}]',
    prev_action='Previous Thought: All fields verified.\nPrevious Action: {"tool": "verify_text_present", "args": {"text": "John Doe"}}\nTool Result: success'
)

# ── [FLOW BLOCKED] TESTS ─────────────────────────────────────────────────────

# FB-1: critical element missing — login button not in DOM
test_inference(
    r"datasets/executor/generated_screenshots/blocked_login_page_missing_step2_1772373359.png",
    "Verify the Login button is present on the login page. If it is missing, the entire login flow is blocked and cannot continue.",
    dom='[{"tag": "input", "selector": "[data-test=\"login-button\"]", "type": "submit", "value": "Login", "in_viewport": true}]'
)

# FB-2: submit button wrong selector — blocks form submission
test_inference(
    r"datasets/executor/generated_screenshots/blocked_form_submit_missing_step3_1772373364.png",
    "Verify the Submit button (#submitButton) is present. Cannot complete form submission without it — flow would be blocked.",
    dom='[{"tag": "a", "selector": "a.router-link", "text": "Text Box", "type": "", "value": "", "in_viewport": true}]'
)

# FB-3: inventory container not loaded after login — blocks checkout
test_inference(
    r"datasets/executor/generated_screenshots/blocked_inventory_not_loaded_step5_1772373367.png",
    "Verify the inventory container loaded. If not visible after login, this is a critical bug blocking the entire checkout flow.",
    dom='[{"tag": "a", "selector": "[data-test=\"item-5-img-link\"]", "type": "", "value": "", "in_viewport": true}, {"tag": "button", "selector": "[data-test=\"add-to-cart-sauce-labs-bolt-t-shirt\"]", "text": "Add to cart", "type": "submit", "in_viewport": true}]'
)

# ── RAISE BUG TESTS ──────────────────────────────────────────────────────────

# RB-1: color mismatch — button should be red but is green
test_inference(
    r"datasets/executor/generated_screenshots/color_fail_button_wrong_color_step2_1772373079.png",
    "Verify Login button color is red (#dc3545) as per the test specification. This assertion is expected to FAIL — button is actually green.",
    dom='[{"tag": "input", "selector": "[data-test=\"login-button\"]", "text": "Login", "type": "submit", "value": "Login", "in_viewport": true}]'
)

# RB-2: wrong selector — #fullName doesn't exist, #userName does
test_inference(
    r"datasets/executor/generated_screenshots/neg_selector_wrong_id_step2_1772373166.png",
    "Verify element #fullName is present. NOTE: this selector does NOT exist on this page.",
    dom='[{"tag": "input", "selector": "#userName", "type": "text", "value": "", "text": "", "in_viewport": true}, {"tag": "input", "selector": "#userEmail", "type": "email", "value": "", "in_viewport": true}]'
)


[Task]: Validate that the Full Name field contains John Doe.
[Response]:
Thought: The field #userName contains John Doe. I will mark this step as passed.
Action: mark_step_pass
Final Answer: The Full Name field contains John Doe. (passed)

[Task]: Validate that the Email field contains john.doe@testmail.com.
[Response]:
Thought: The field is visible and contains the expected value. This step passes.
Action: {"tool": "verify_input_value", "args": {"selector": "#userEmail", "expected": "john.doe@testmail.com"}}
Final Answer: Verified: Field contains john.doe@testmail.com.
[{"status": "pass", "tool": "verify_input_value", "args": {"selector": "#userEmail", "expected": "john.doe@testmail.com"}}]

[Task]: Verify that the submitted output section shows John Doe.
[Response]:
Thought: I need to verify the submitted data is displayed in the lower section. The target element is #submit-output.
Action: {"tool": "verify_element_visible", "args": {"selector": "#submit-output"}}
Final Answer: Verif

'Thought: The selector #fullName is invalid and will not match any element on this page. This step is blocked.\nAction: {"tool": "raise_bug_ticket", "args": {"title": "Selector #fullName is invalid", "severity": "Critical"}}\nFinal Answer: Bounding box calculation failed or DOM selector is invalid.'